# 07 · SICD → analysis-ready GeoTIFF: geocoding the complex product

Every other notebook in this gallery reaches for the `GEC` asset — the
detected, **already geocoded** cloud-optimized GeoTIFF Umbra ships for you. The
`SICD` is the other half of the archive: the full-resolution *complex* product,
the one you want for interferometry, sub-pixel work, or your own detector. But
it lives in the radar **slant plane** — its pixels are range/azimuth samples,
not lon/lat — so it does not open on a map, in QGIS, or in the
`xarray`/`rioxarray` stack without first being geocoded through the sensor's own
projection model. That is exactly the "same 500 lines of glue" that makes people
give up on complex SAR.

`umbra convert` closes it in one call. This notebook takes one SICD from the
open archive, detects its amplitude in the slant plane (the fastest look), then
geocodes it onto a north-up **EPSG:4326 cloud-optimized GeoTIFF** that drops
straight onto a web map — and checks that it lands where the scene really is.

> Needs the `[convert]` extra: `pip install "umbra-py[convert]"` (pulls in
> `sarpy` + `rasterio`).

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Find a SICD scene

A one-day window keeps the search deterministic. Most tasks carry a `SICD`
alongside the `GEC`; we take the first one. `Centerfield, Utah` is a small,
fast scene — the same site the other notebooks use — so the download and
conversion below stay quick.

In [ ]:
from umbra_py import UmbraCatalog

scene = next(
    it
    for it in UmbraCatalog().search(
        start="2024-12-11", end="2024-12-11", product_types=["SICD"], limit=5
    )
    if "SICD" in it.available_assets
)

assert "SICD" in scene.available_assets
print(scene.task, "·", scene.datetime)
print("resolution (m):", tuple(round(r, 2) for r in scene.resolution))
print("polarizations:", scene.polarizations)

## 2 · Download the complex product

The `GEC` is a COG the other notebooks stream one window at a time over HTTP
range reads. The `SICD` is different: it is a complex NITF that has to be read
*whole* to geocode, so we download it once (a few hundred MB) and convert it
locally. There is no shortcut here — the complex data is the point.

In [ ]:
import os
import tempfile

from umbra_py import download_url

work = tempfile.mkdtemp(prefix="umbra-sicd-")
sicd_path = download_url(scene.asset_href("SICD"), os.path.join(work, "scene.nitf"))

assert os.path.getsize(sicd_path) > 0
print("downloaded", round(os.path.getsize(sicd_path) / 1e6, 1), "MB ->", sicd_path)

## 3 · Detect amplitude — a look in the slant plane

The fastest thing you can do with a SICD is detect its amplitude
(`|complex|`) and write a GeoTIFF. `sicd_to_amplitude_geotiff` does that — but
the output is still in the **slant plane**: it carries no geotransform, so it
opens as an *image*, not a map. Note the CRS below is `None`: these pixels are
radar rows and columns, not geographic coordinates.

In [ ]:
import rasterio

from umbra_py import sicd_to_amplitude_geotiff

amp_path = sicd_to_amplitude_geotiff(sicd_path, os.path.join(work, "amplitude.tif"))
with rasterio.open(amp_path) as ds:
    amp_crs = ds.crs
    print("slant-plane amplitude:", ds.width, "x", ds.height, "·", ds.dtypes[0], "· CRS:", ds.crs)

# No map geometry yet — the axes are range/azimuth, not lon/lat.
assert amp_crs is None

## 4 · Geocode onto a map-ready COG

`sicd_to_geocoded_cog` warps that amplitude onto a north-up EPSG:4326
cloud-optimized GeoTIFF using SICD's **own image-projection model** — a lattice
of ground control points sampled across the image — so the sensor geometry, not
a naive corner-stretch, places every pixel. The result is a real COG (with
overviews for fast zoomed reads) that opens on a web map, in QGIS, or as a
georeferenced array with no sidecar.

In [ ]:
from umbra_py import sicd_to_geocoded_cog

cog_path = sicd_to_geocoded_cog(sicd_path, os.path.join(work, "geocoded.tif"))
with rasterio.open(cog_path) as ds:
    cog_epsg = ds.crs.to_epsg()
    bounds = ds.bounds
    overviews = ds.overviews(1)
    print("geocoded COG:", ds.width, "x", ds.height, "· CRS:", ds.crs)
    print("bounds:", tuple(round(v, 4) for v in bounds))
    print("overviews:", overviews)

# It now carries real geographic geometry — EPSG:4326 — and COG overviews.
assert cog_epsg == 4326
assert overviews, "a COG carries overviews for fast zoomed reads"

## 5 · It lands where the scene really is

Geocoding is only correct if the raster ends up in the *right place*. The
projected footprint should sit on top of the acquisition's own bounding box
from the catalog — proof the projection model placed the pixels on Earth, not
just somewhere plausible.

In [ ]:
gminx, gminy, gmaxx, gmaxy = bounds.left, bounds.bottom, bounds.right, bounds.top
fminx, fminy, fmaxx, fmaxy = scene.bbox

overlaps = not (gmaxx < fminx or gminx > fmaxx or gmaxy < fminy or gminy > fmaxy)
print("catalog footprint :", tuple(round(v, 4) for v in scene.bbox))
print("geocoded footprint:", (round(gminx, 4), round(gminy, 4), round(gmaxx, 4), round(gmaxy, 4)))

assert overlaps, "the geocoded scene lands on its catalog footprint"
print("\ngeocoded SICD lands on the acquisition's footprint")

## Where next

- **Terrain-orthorectify over relief.** Flat-earth geocoding (the default)
  places pixels on the scene's height-above-ellipsoid plane — exact over flat
  terrain, but it mislocates hilltops and valley floors. Pass
  `sicd_to_geocoded_cog(..., dem="auto")` (or `umbra convert SRC DST --dem
  auto`) and it fetches the covering Copernicus GLO-30 tiles and walks each
  control point onto the terrain surface. Add `--geoid auto` for survey-grade
  vertical referencing and `--rtc` to flatten the brightness modulation slopes
  add.
- **From the CLI.** `umbra convert scene.nitf out.tif` mirrors this notebook;
  `--slant-plane` emits the quick amplitude look instead of the geocoded COG.
- **Feed it downstream.** The geocoded COG is an ordinary EPSG:4326 raster, so
  `rioxarray`, `umbra chips` (notebook 05), and QGIS all take it directly. When
  a task already carries a `GEC`, that is the same map-ready product Umbra
  geocoded for you — `to_xarray(scene, asset="GEC")` loads it in one line.

*Contains Umbra open data, licensed under CC BY 4.0.*